# Amazon ML Challenge 2026: Entity Resolution Suite
### Google Colab GPU/CPU Runner with Google Drive Persistence & Auto-Resume

**Running in Antigravity IDE via Google Colab Extension**:
1. Open Command Palette (`Ctrl+Shift+P` / `Cmd+Shift+P`).
2. Select `Google Colab: Connect to a Colab runtime` (or click Kernel selector top-right -> Google Colab).
3. Choose runtime accelerator: **L4 GPU** (or T4 GPU) with High-RAM.
4. Run all cells sequentially!

**Key Pipeline Architecture**:
- **Dedicated Run Isolation**: Each run writes to `output/runs/<run_id>/` and `drive/runs/<run_id>/`—prior outputs are never overwritten!
- **Speed-Optimized Inference**: TF-IDF cosine candidate pre-pruning + batched gatekeeper inference cuts runtime by >65%.
- **Drive Persistence**: Real-time mirroring to Google Drive checkpoints (`/content/drive/MyDrive/amazon-ml-challenge-2026`).
- **Official Macro F0.5 Metric**: Direct threshold optimization weighted towards precision (beta=0.5).
- **Memory-Safe Audit**: Built-in format verification with `utils/validate_submission.py`.

In [ ]:
# 1. Verify GPU availability and system resources
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):" || true

In [ ]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Initialize Google Drive directory structure for amazon-ml-challenge-2026
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-ml-challenge-2026")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_OUTPUT = DRIVE_ROOT / "output"

for folder in [DRIVE_ROOT, DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir   : {DRIVE_DATA}")
print(f"    - Weights Dir    : {DRIVE_WEIGHTS}")
print(f"    - Checkpoints Dir: {DRIVE_CHECKPOINTS}")
print(f"    - Output Dir     : {DRIVE_OUTPUT}")

In [ ]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull || true

In [ ]:
# 5. Install required packages (rapid string matching, lightgbm, etc.)
!python install.py

In [ ]:
# 6. Dataset Linking, Shards Setup & Verification
# Google Drive datasets (/content/drive/MyDrive/amazon-ml-challenge-2026/data/)
# are symlinked to local data/ for zero-copy high-throughput I/O.
import os
from pathlib import Path

!mkdir -p data
drive_data = Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data")
if drive_data.exists():
    for item in drive_data.iterdir():
        target = Path(f"data/{item.name}")
        if not target.exists():
            os.symlink(str(item), str(target))
            print(f"[✓] Linked {item.name} from Google Drive -> data/{item.name}")

# Check available dataset paths
print("\n--- AVAILABLE DATASETS CHECK ---")
for p in ["data/shards/train", "data/shards/test", "dataset_split/train", "dataset_split/test"]:
    if Path(p).exists():
        print(f"  [READY] Found dataset: {p}")
    else:
        print(f"  [INFO] Path not local: {p} (will look in Google Drive if uploaded)")

In [ ]:
# 7. Fast Smoke Test (~10 seconds)
# Verifies normalization, candidate blocking, classifier, decision layer & audit end-to-end
!python scripts/run_pipeline.py \
    --train-dir dataset_split/train \
    --test-dir dataset_split/test \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --smoke-test

In [ ]:
# 9. Full Pipeline Execution: Training, Blocking & Chunked Inference (Crash-Resilient with Auto-Resume)
# Automatically uses GPU if available, throttles RAM safely, and checkpoints every chunk to Google Drive.
# Outputs are saved in dedicated run folders (output/runs/<run_id>/) so past runs are NEVER overwritten.
# If interrupted or timed out, re-running this cell resumes from the exact last completed batch!

import os
import time

train_path = "data/shards/train" if os.path.exists("data/shards/train") else "dataset_split/train"
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

run_id = f"prod_l4_fast_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"Starting Run ID: {run_id}")
print(f"Train Dataset:   {train_path}")
print(f"Test Dataset:    {test_path}")

# Check if candidate_pairs already exists in Google Drive or previous run
cands_arg = ""
drive_cand_path = "/content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv"
if os.path.isfile(drive_cand_path):
    print(f"[*] Found pre-computed candidate_pairs.tsv in Google Drive: {drive_cand_path}")
    cands_arg = f"--candidate-pairs-path {drive_cand_path}"

!python scripts/run_pipeline.py \
    --train-dir $train_path \
    --test-dir $test_path \
    --run-id $run_id \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --device auto \
    --batch-size 10000 \
    --n-jobs -1 \
    $cands_arg


In [ ]:
# 10. Run Official Submission Validator on Generated Files
# Validates format, ID compliance, and required columns for competition submission (<50MB RAM, ~15s)
import os
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

!python utils/validate_submission.py \
    --matching output/latest/matching_results.tsv \
    --test-dir $test_path


In [ ]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/latest/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- Latest Run Output Files ---")
!ls -lh output/latest/
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/latest/matching_results.tsv


In [ ]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- First 5 rows of candidate_pairs.tsv ---")
!head -n 6 output/candidate_pairs.tsv
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/matching_results.tsv